# Stage 04b: Train/Test Split

**Purpose:** Split data by fold

**Outputs:** data/04b_train.parquet, data/04b_test.parquet

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_coll/v1"


In [3]:
import pandas as pd
import yaml
import os
import sys
import gc
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 04b: TRAIN/TEST SPLIT")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 04b: TRAIN/TEST SPLIT
########################################


In [4]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

# Get current machine ID and output path
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'  # current.pc has '3', config key is 'PC3'
output_base = cfg['machines'][pc_id]['paths']['output_path']

In [5]:
checkpoint_02 = f"{output_base}/data/02_conditioned.parquet"
data = pd.read_parquet(checkpoint_02)
print(f"\n* Loaded: {data.shape}")

# Merge PCA features if enabled
pca_cfg = cfg.get('feature_engineering', {}).get('pca', {})
pca_enabled = pca_cfg.get('enabled', False)

if pca_enabled:
    pca_file = f"{output_base}/data/04a_pca_features.parquet"
    if os.path.exists(pca_file):
        pca_features = pd.read_parquet(pca_file)
        join_key = cfg['data']['join_key']
        data = data.merge(pca_features, on=join_key, how='left')
        n_pca_cols = len(pca_features.columns) - 1
        print(f"  [OK] Merged {n_pca_cols} PCA features")

# Validate critical columns exist
critical_cols = [
    cfg['data']['join_key'],
    cfg['data']['fold_column'],
    cfg['experiment']['target'],
    cfg['experiment']['exposure']
]
missing = [c for c in critical_cols if c not in data.columns]
if missing:
    raise ValueError(f"Missing critical columns: {missing}")

print(f"\n* Final shape: {data.shape}")
print(f"* Critical columns validated: {', '.join(critical_cols)}")


* Loaded: (18939238, 112)


  [OK] Merged 10 PCA features

* Final shape: (26193532, 122)
* Critical columns validated: vin_date, fold, pp_coll, ee_coll_imps


In [6]:
# Split by fold
mode = cfg['run_mode']
fold_col = cfg['data']['fold_column']
test_folds = cfg[mode]['test_folds']
train_folds = cfg[mode]['train_folds']

train_data = data[data[fold_col].isin(train_folds)].copy()
test_data = data[data[fold_col].isin(test_folds)].copy()

print(f"\n* Train: {train_data.shape}")
print(f"* Test: {test_data.shape}")

del data
gc.collect()


* Train: (18709834, 122)
* Test: (7483698, 122)


0

In [7]:
# Validate outputs before saving
from utils import validate_critical_columns
validate_critical_columns(train_data, cfg, 'Stage 04 Train Output')
validate_critical_columns(test_data, cfg, 'Stage 04 Test Output')

# Save splits
train_file = f"{output_base}/data/04b_train.parquet"
test_file = f"{output_base}/data/04b_test.parquet"

train_data.to_parquet(train_file)
test_data.to_parquet(test_file)

print(f"\n* Saved: {train_file}")
print(f"* Saved: {test_file}")

[Stage 04 Train Output] [OK] Critical columns validated: vin_date, fold, pp_coll, ee_coll_imps
[Stage 04 Test Output] [OK] Critical columns validated: vin_date, fold, pp_coll, ee_coll_imps



* Saved: output/car_coll/v1/data/04b_train.parquet
* Saved: output/car_coll/v1/data/04b_test.parquet


In [8]:
print("\n########################################")
print("# STAGE 04b: COMPLETE")
print("########################################")


########################################
# STAGE 04b: COMPLETE
########################################
